In [1]:
import os
from datetime import datetime

import torch
from torch import nn
from torchvision import datasets
import torchvision.transforms.v2 as transforms_v2
from torchvision.io import read_image
from torch.utils.data import DataLoader
from torch.utils.tensorboard import SummaryWriter
from torcheval.metrics.functional import (
    multiclass_accuracy,
    multiclass_f1_score
)
import torch.nn.functional as F

In [2]:
from roboflow import Roboflow
rf = Roboflow(api_key="D2P3bCoryjlMEaJoExwn")
project = rf.workspace("wrkspc-gi0hz").project("cloud-classification-mf91q")
version = project.version(7)
dataset = version.download("tensorflow")

loading Roboflow workspace...
loading Roboflow project...


In [3]:
CLASS_NAMES = [
    "altocumulus",
    "altostratus",
    "cirrocumulus",
    "cirrostratus",
    "cirrus",
    "cumulonimbus",
    "cumulus",
    "nimbostratus",
    "stratocumulus",
    "stratus",
]

In [4]:
IMG_SIZE = (224, 224)

def prepare_dataset(records, image_path):
    images = []
    targets = []
    labels = []

    for _, row in records.iterrows():
        filename, img_w, img_h, class_name, xmin, ymin, xmax, ymax = row
        
        fullpath = os.path.join(image_path, filename)
        img = read_image(fullpath)
        
        # Normalize
        xmin = xmin / img_w
        ymin = ymin / img_h
        xmax = xmax / img_w
        ymax = ymax / img_h

        '''
        # Convert to CXCYWH
        center_x = (xmin + xmax) / 2
        center_y = (ymin + ymax) / 2
        box_w = xmax - xmin
        box_h = ymax - ymin
        '''
        
        images.append(img)
        targets.append((xmin, ymin, xmax, ymax))
        labels.append(CLASS_NAMES.index(class_name))

    return images, targets, labels

In [5]:
import os
import pandas as pd

# Train set
TRAINING_CSV_FILE = 'Cloud-Classification-7/train/_annotations.csv'
TRAINING_IMAGE_DIR = 'Cloud-Classification-7/train'

training_image_records = pd.read_csv(TRAINING_CSV_FILE)

train_image_path = os.path.join(os.getcwd(), TRAINING_IMAGE_DIR)

train_images, train_targets, train_labels = prepare_dataset(training_image_records, train_image_path)

# Validate set
VALIDATING_CSV_FILE = 'Cloud-Classification-7/valid/_annotations.csv'
VALIDATING_IMAGE_DIR = 'Cloud-Classification-7/valid'

validating_image_records = pd.read_csv(VALIDATING_CSV_FILE)

valid_image_path = os.path.join(os.getcwd(), VALIDATING_IMAGE_DIR)

valid_images, valid_targets, valid_labels = prepare_dataset(validating_image_records, valid_image_path)

# Testing set
TESTING_CSV_FILE = 'Cloud-Classification-7/test/_annotations.csv'
TESTING_IMAGE_DIR = 'Cloud-Classification-7/test'

testing_image_records = pd.read_csv(TESTING_CSV_FILE)

test_image_path = os.path.join(os.getcwd(), TESTING_IMAGE_DIR)

test_images, test_targets, test_labels = prepare_dataset(testing_image_records, test_image_path)

In [6]:
print(test_targets[0])

(0.0125, 0.0453125, 0.9390625, 0.965625)


In [7]:
train_images = torch.stack(train_images).float() / 255.0
train_targets = torch.tensor(train_targets, dtype=torch.float32)
train_labels = torch.tensor(train_labels, dtype=torch.long)

valid_images = torch.stack(valid_images).float() / 255.0
valid_targets = torch.tensor(valid_targets, dtype=torch.float32)
valid_labels = torch.tensor(valid_labels, dtype=torch.long)

test_images = torch.stack(test_images).float() / 255.0
test_targets = torch.tensor(test_targets, dtype=torch.float32)
test_labels = torch.tensor(test_labels, dtype=torch.long)

In [8]:
num_classes = len(CLASS_NAMES)

#create the common input layer
input_shape = (IMG_SIZE[0], IMG_SIZE[1], 3)

In [9]:
from torch import nn

losses = {
    "cl_head": nn.CrossEntropyLoss(),  # Loss for class prediction
    "bb_head": nn.MSELoss() #bbox_loss # Loss for bounding box prediction
}

In [10]:
trainTargets = {
    "cl_head": train_labels,
    "bb_head": train_targets
}
validTargets = {
    "cl_head": valid_labels,
    "bb_head": valid_targets
}

In [11]:
from torchvision.models import mobilenet_v2, MobileNet_V2_Weights


if torch.cuda.is_available():
    device = "cuda"
elif torch.backends.mps.is_available():
    device = "mps"
else:
    device = "cpu"

print("Using device:", device)

class MobileNetMultiHead(nn.Module):
    def __init__(self, num_classes):
        super().__init__()
        weights = MobileNet_V2_Weights.DEFAULT
        model = mobilenet_v2(weights=weights)
        self.backbone = nn.Sequential(*list(model.children())[:-1])

        # backbone (remove classifier)
        self.features = model.features

        # global average pooling
        self.pool = nn.AdaptiveAvgPool2d((3, 3))

        in_features = model.classifier[1].in_features
        
        # classification head
        self.cl_head = nn.Linear(in_features, num_classes)

        # bounding box head
        self.bb_head = nn.Linear(in_features, 4)

    def forward(self, x):

        # equivalent to preprocess_input must be done in transforms

        x = self.backbone(x)

        x = self.pool(x)
        x = torch.flatten(x, 1)

        
        class_output = torch.softmax(self.cl_head(x), dim=1)
        bbox_output = torch.sigmoid(self.bb_head(x))
        '''
        class_output = self.cl_head(x)
        bbox_output = self.bb_head(x)
        '''

        return class_output, bbox_output


from torchvision.models import resnet18, ResNet18_Weights

class MultiTaskModel(nn.Module):
    def __init__(self, num_classes=3):
        super(MultiTaskModel, self).__init__()
        
        # Load ResNet backbone
        weights = ResNet18_Weights.DEFAULT
        resnet = resnet18(weights=weights)
        self.backbone = nn.Sequential(*list(resnet.children())[:-1])  # Remove fully connected layer

        # The output size of from self.backbone
        n_features = resnet.fc.in_features

        # TODO: Classification head
        self.classifier = nn.Linear(n_features, num_classes) # YOUR CODE HERE

        # TODO: Localization head (bounding box regression)
        self.regressor = nn.Linear(n_features, 4) # YOUR CODE HERE

    def forward(self, x):
        out = self.backbone(x)
        out = torch.flatten(out, 1)  # Flatten the output

        # TODO: Model output
        class_out = self.classifier(out) # YOUR CODE HERE for classification output
        bbox_out = self.regressor(out) # YOUR CODE HERE for bounding box regression output

        return class_out, bbox_out

Using device: cuda


In [12]:
model = MultiTaskModel(num_classes=num_classes)
model = model.to(device)

'''
fine_tune_at = 8
ct = 0
for child in model.children():
    ct += 1
    if ct < fine_tune_at:
        for param in child.parameters():
            param.requires_grad = False
        print(f"{child} ({ct}): {False}")
    else:
        for param in child.parameters():
            param.requires_grad = True
        print(f"{child} ({ct}): {True}")
'''

'\nfine_tune_at = 8\nct = 0\nfor child in model.children():\n    ct += 1\n    if ct < fine_tune_at:\n        for param in child.parameters():\n            param.requires_grad = False\n        print(f"{child} ({ct}): {False}")\n    else:\n        for param in child.parameters():\n            param.requires_grad = True\n        print(f"{child} ({ct}): {True}")\n'

In [13]:
def train_one_epoch(
        dataloader, model, losses, optimizer, 
        epoch, device, writer, log_step_interval=50
    ):
    size = len(dataloader.dataset)
    model.train()
    running_loss = 0

    for i, (X, y, bboxes) in enumerate(dataloader):

        X = X.to(device)
        y = y.to(device)
        bboxes = bboxes.to(device)

        optimizer.zero_grad()

        cl_pred, bb_pred = model(X)

        cl_loss = losses["cl_head"](cl_pred, y)
        bb_loss = losses["bb_head"](bb_pred, bboxes)

        total_loss = cl_loss + bb_loss

        total_loss.backward()
        optimizer.step()

        running_loss += total_loss.item()
        if (i+1) % log_step_interval == 0:
            print(f"Epoch {epoch+1}, Step {i+1}/{len(dataloader)}, Loss: {total_loss.item():.4f} (Class: {cl_loss.item():.4f}, BBox: {bb_loss.item():.4f})")
            writer.add_scalar('Loss/train', total_loss.item(), epoch * len(dataloader) + i)




def test(dataloader, model, losses, device):
    num_batches = len(dataloader)
    model.eval()
    cl_loss, bb_loss = 0, 0
    y_preds, y_trues = [], []
    bbox_preds, bbox_trues = [], []

    with torch.no_grad():
        for i, (X, y, bboxes) in enumerate(dataloader):

            X = X.to(device)
            y = y.to(device)
            bboxes = bboxes.to(device)

            cl_pred, bb_pred = model(X)

            cl_loss += losses["cl_head"](cl_pred, y)
            bb_loss += losses["bb_head"](bb_pred, bboxes)

            y_preds.append(cl_pred.argmax(1))
            y_trues.append(y)
            bbox_preds.append(bb_pred)
            bbox_trues.append(bboxes)

    y_preds = torch.cat(y_preds)
    y_trues = torch.cat(y_trues)
    bbox_preds = torch.cat(bbox_preds)
    bbox_trues = torch.cat(bbox_trues)

    cl_loss /= num_batches
    bb_loss /= num_batches
    return cl_loss, bb_loss, y_preds, y_trues, bbox_preds, bbox_trues

In [14]:
from datetime import datetime
from torch.utils.tensorboard import SummaryWriter

learning_rate = 1e-5
batch_size = 4   
epochs = 10           
optimizer = torch.optim.AdamW(model.parameters(), lr=learning_rate)
writer = SummaryWriter(f'./runs/trainer_{model._get_name()}_{datetime.now().strftime("%Y%m%d-%H%M%S")}')

In [15]:
from torch.utils.data import TensorDataset, DataLoader

train_ds = TensorDataset(train_images, train_labels, train_targets)
valid_ds = TensorDataset(valid_images, valid_labels, valid_targets)
test_ds = TensorDataset(test_images, test_labels, test_targets)

train_dl = DataLoader(train_ds, batch_size=32, shuffle=True)
valid_dl = DataLoader(valid_ds, batch_size=32, shuffle=False)
test_dl = DataLoader(test_ds, batch_size=32, shuffle=False)

In [16]:
batch = next(iter(train_dl))

print(type(batch))
print(len(batch))

for i, item in enumerate(batch):
    print(i, type(item))

<class 'list'>
3
0 <class 'torch.Tensor'>
1 <class 'torch.Tensor'>
2 <class 'torch.Tensor'>


In [17]:
from torcheval.metrics.functional import (
    multiclass_accuracy,
    multiclass_f1_score
)

best_vloss = 100000.
for epoch in range(epochs):
    print(f"Epoch {epoch+1} / {epochs}")
    train_one_epoch(train_dl, model, losses, optimizer, epoch, device, writer, log_step_interval=1)

    # Compute train & validation loss
    train_loss, train_bbox_loss, train_y_preds, train_y_trues, train_bbox_preds, train_bbox_trues = test(
        train_dl, model, losses, device
    )
    val_loss, val_bbox_loss, val_y_preds, val_y_trues, val_bbox_preds, val_bbox_trues = test(
        valid_dl, model, losses, device
    )

    # Compute classification metrics
    train_accuracy = multiclass_accuracy(train_y_preds, train_y_trues).item()
    train_f1 = multiclass_f1_score(train_y_preds, train_y_trues).item()
    val_accuracy = multiclass_accuracy(val_y_preds, val_y_trues).item()
    val_f1 = multiclass_f1_score(val_y_preds, val_y_trues).item()

    # Compute bounding box MSE
    train_bbox_mse = F.mse_loss(train_bbox_preds, train_bbox_trues).item()
    val_bbox_mse = F.mse_loss(val_bbox_preds, val_bbox_trues).item()

    # Log training performance
    writer.add_scalars('Train vs. Valid/loss', 
        {'train': train_loss, 'valid': val_loss}, 
        epoch)
    writer.add_scalars('Train vs. Valid/bbox_mse', 
        {'train': train_bbox_mse, 'valid': val_bbox_mse}, 
        epoch)
    writer.add_scalars('Train vs. Valid/acc', 
        {'train': train_accuracy, 'valid': val_accuracy}, 
        epoch)
    writer.add_scalars('Train vs. Valid/f1', 
        {'train': train_f1, 'valid': val_f1}, 
        epoch)

    # Save the best model
    if val_loss < best_vloss:
        best_vloss = val_loss
        torch.save(model.state_dict(), 'model_best_vloss.pth')
        print('Saved best model to model_best_vloss.pth')

print("Training Complete!")

Epoch 1 / 10
Epoch 1, Step 1/49, Loss: 3.0033 (Class: 2.3009, BBox: 0.7024)
Epoch 1, Step 2/49, Loss: 3.0346 (Class: 2.3306, BBox: 0.7041)
Epoch 1, Step 3/49, Loss: 3.2035 (Class: 2.4803, BBox: 0.7231)
Epoch 1, Step 4/49, Loss: 3.0270 (Class: 2.3069, BBox: 0.7201)
Epoch 1, Step 5/49, Loss: 3.0647 (Class: 2.3860, BBox: 0.6787)
Epoch 1, Step 6/49, Loss: 2.9425 (Class: 2.2706, BBox: 0.6718)
Epoch 1, Step 7/49, Loss: 3.0477 (Class: 2.3664, BBox: 0.6813)
Epoch 1, Step 8/49, Loss: 3.0378 (Class: 2.3520, BBox: 0.6858)
Epoch 1, Step 9/49, Loss: 2.8656 (Class: 2.2121, BBox: 0.6535)
Epoch 1, Step 10/49, Loss: 2.9497 (Class: 2.3241, BBox: 0.6256)
Epoch 1, Step 11/49, Loss: 2.9812 (Class: 2.3060, BBox: 0.6752)
Epoch 1, Step 12/49, Loss: 2.7732 (Class: 2.1617, BBox: 0.6115)
Epoch 1, Step 13/49, Loss: 2.9212 (Class: 2.3132, BBox: 0.6081)
Epoch 1, Step 14/49, Loss: 2.8445 (Class: 2.2544, BBox: 0.5900)
Epoch 1, Step 15/49, Loss: 2.8574 (Class: 2.2428, BBox: 0.6147)
Epoch 1, Step 16/49, Loss: 2.8959 (C

In [18]:
model_best = MultiTaskModel(num_classes=len(CLASS_NAMES))
model_best = model_best.to(device)
model_best.load_state_dict(torch.load("model_best_vloss.pth"))

# Evaluate on the test set
test_loss, test_bbox_loss, test_y_preds, test_y_trues, test_bbox_preds, test_bbox_trues = test(
    test_dl, model, losses, device
)

# Compute test classification metrics
test_accuracy = multiclass_accuracy(test_y_preds, test_y_trues).item()
test_f1 = multiclass_f1_score(test_y_preds, test_y_trues).item()

# Compute bounding box MSE
test_bbox_mse = F.mse_loss(test_bbox_preds, test_bbox_trues).item()

print(f"\nTest Results:")
print(f"Classification Loss: {test_loss:.4f}")
print(f"Bounding Box MSE: {test_bbox_mse:.4f}")
print(f"Accuracy: {test_accuracy:.2f}%")
print(f"F1 Score: {test_f1:.2f}")


Test Results:
Classification Loss: 0.7793
Bounding Box MSE: 0.0282
Accuracy: 0.75%
F1 Score: 0.75


In [19]:
print("pred:", test_bbox_preds[0])
print("target:", test_bbox_trues[0])

pred: tensor([0.0129, 0.0794, 0.9929, 0.7652], device='cuda:0')
target: tensor([0.0125, 0.0453, 0.9391, 0.9656], device='cuda:0')
